# Session 2: Validation Validation Validation
Now we've got to the stage of being able to make a passable chatbot, we can start to take advantage of the real benefits of using PydanticAI that help integrate it's systems into larger works - output validation!

In this session we start using the result_type argument, implement logical guards with @field_validator, and understand the Retry Loop.

In [ ]:
import os
from datetime import datetime
from enum import Enum
from typing import List, Optional

from pydantic import BaseModel, Field, field_validator
from pydantic_ai import Agent, ModelRetry, RunContext
from pydantic_settings import BaseSettings, SettingsConfigDict


class Settings(BaseSettings):
    model_config = SettingsConfigDict(env_file=".env")
    openai_api_key: str
    open_ai_default_model: str = "openai:gpt-5-nano"


settings = Settings()

## Part 1: Typed Outputs
First we started asking models "pretty please return JSON or I'll lose my job!", then we started providing JSON schemas as part of the model context. Now most LLMs providers are capable of going one step further and restricting model outputs to guarantee the output matches the schema - by actually restricting the output tokens.

In [ ]:
# 1. Define the Schema
class Genre(str, Enum):
    ACTION = "action"
    ADVENTURE = "adventure"
    COMEDY = "comedy"
    DRAMA = "drama"
    HORROR = "horror"
    ROMANCE = "romance"
    SCI_FI = "sci-fi"
    THRILLER = "thriller"
    FANTASY = "fantasy"
    DOCUMENTARY = "documentary"
    ANIMATION = "animation"
    CRIME = "crime"
    MYSTERY = "mystery"
    WAR = "war"
    WESTERN = "western"


class MovieExtraction(BaseModel):
    title: str
    director: str
    year: int
    genres: List[Genre]
    short_summary: str = Field(description="A one-sentence summary")


# 2. Define the Agent with the Schema
extraction_agent = Agent(
    settings.open_ai_default_model,
    output_type=MovieExtraction,  # <--- The Magic
    system_prompt="Extract movie details from the user text.",
)

In [ ]:
# 3. Run
text = "I just watched Inception by Nolan. It came out in 2010. Mind bending sci-fi thriller."
result = await extraction_agent.run(text)

# 4. Result is a validated Object, not a dict or string
print(f"Type: {type(result.output)}")
print(f"Movie: {result.output.title} ({result.output.year})")
print(f"Genres: {result.output.genres}")

In the above snippet, there are several abstractions worth examining.

#### What is an `Enum`?

An `Enum` (Enumeration) defines a fixed set of constant values. It acts as the "source of truth" for what options are allowed in your program. By inheriting from `str` (as in `str`, `Enum`), you ensure these constants behave like strings when sent to the LLM (JSON-serializable) but act like strict, type-safe objects in your code.
* **Access**: The variables are name-spaced to keep them organised. You access them via dot notation (Genre.ACTION) rather than using raw strings loosely in your code.

#### Why is `Genre` not called?

Because `Genre` acts as a catalog of existing constants, you never "call" it or create a new instance (e.g., `Genre()`). It works as a group of constants, not as a factory creating objects. Values like `Genre.ACTION` already exist and are ready to be selected.

#### What is PydanticAI doing to the LLM here?

PydanticAI manages the entire data lifecycle, not just validation:
1. Translation: It converts your Python class (including the `Genre` enum) into a standardized JSON Schema.
2. Prompting: It injects this schema into either the system prompt (effectively, giving the LLM a strict rulebook, "You must answer using this exact structure") or the request itself for compatible LLMs such as OpenAI's GPT series where the schema will be strictly enforced by restricting the tokens the model can output (see the README reading list for more info).
3. Validation & Parsing: When the LLM replies with a string like "action", Pydantic validates it against your "source of truth" and converts it back into the Python object Genre.ACTION.
    * **Result**: You don't get a raw string; you get a type-safe `Enum` object you can use in logic, for instance:



In [ ]:
if result.output.genres[0] == Genre.SCI_FI:
    print(f"It is indeed {Genre.SCI_FI.value}!")

## Part 2: Enforcing Logic: Validators as Guardrails
Getting the structure right is easy (JSON Mode). Getting the logic right is hard. What if a film review extractor hallucinates a year like 2030? Or returns a confidence score of 150%?

Pydantic to the rescue again. We use standard Pydantic @field_validator. If a validation error is raised, PydanticAI catches it, sends the error message back to the LLM, and asks it to try again.

The Retry Loop Visualized:  
* `Agent`: "Here is the data: {year: 1800}"
* Pydantic: ValidationError: Year must be > 1900
* PydanticAI: (Intercepts error, raise ModelRetry error) -> Sends user message: "Error: Year must be > 1900. Fix this."
* `Agent`: "Apologies. {year: 1900}" -> Success

In [ ]:
# 1. Define MovieExtractionV2 with validation
class MovieExtractionV2(BaseModel):
    title: str
    director: str
    year: Optional[int] = None
    genres: List[Genre]
    short_summary: str = Field(description="A one-sentence summary")

    @field_validator("year")
    @classmethod
    def validate_year_not_future(cls, v: Optional[int]) -> Optional[int]:
        """Ensure the film wasn't released in the future."""
        if v is None:
            return v
        current_year = datetime.now().year
        if v > current_year:
            raise ValueError(
                f"Year {v} is in the future. The current year is {current_year}. If the release year is an estimate in"
                "the future return None."
            )
        return v


# 2. Create agent with V2 schema
extraction_agent_v2 = Agent(
    settings.open_ai_default_model,
    output_type=MovieExtractionV2,
    system_prompt="Extract movie details from the user text. Be accurate with release years.",
)

# 3. Test with a normal case (should work)
print("=== Test 1: Valid movie (past year) ===")
text1 = "I just watched The Matrix by the Wachowskis. It came out in 1999. Groundbreaking sci-fi action film."
result1 = await extraction_agent_v2.run(text1)
print(f"Movie: {result1.output.title} ({result1.output.year})")
print("Validation passed! ✓\n")

# 4. Test with a text that might cause hallucination (the validator will catch it)
print("=== Test 2: Potential future year hallucination ===")
print("Note: If the model tries to return a future year, the validator will catch it")
print("and PydanticAI will automatically retry with the error message.\n")

text2 = "I'm excited about the upcoming sequel that will be released in 2037."
result2 = await extraction_agent_v2.run(text2)
print(f"Movie: {result2.output.title} ({result2.output.year})")
print("Validation passed! ✓")

In [ ]:
from rich import print as rprint  # Alex - TIL you could do this!

# Let's inspect the messages list to verify it's just a list of objects
rprint(result2.all_messages())

## Part 3: Type Descriptors Guide LLM Behavior
PydanticAI doesn't just use your Pydantic models for validation - it also passes the field descriptions and type information directly to the LLM. This means your `Field(description=...)` annotations become part of the model's instructions, helping it understand what you want *before* it generates output.

**Key Insight:** Better field descriptions = Better outputs, fewer retries.

The LLM sees:
- Field names and types
- Field descriptions (from `Field(description=...)`)
- Enum values (for Enum fields)
- Required vs optional fields

This contextual information helps the model generate more accurate outputs on the first try. They are also important if there are multiple possible interpretations of certain fields.


In [ ]:
# Example 2: Rich descriptions (better guidance for LLM)
class MovieExtractionRich(BaseModel):
    title: str = Field(description="The official title of the film")
    director: str = Field(description="Full name of the primary director")
    year: int = Field(description="An estimate of the year the film is set in")
    genres: List[Genre] = Field(description="List of 1-3 primary genres that best categorize the film")
    short_summary: str = Field(
        description="A concise one-sentence summary (15-25 words) that captures the film's core premise without spoilers"
    )


agent_rich = Agent(
    settings.open_ai_default_model,
    output_type=MovieExtractionRich,
    system_prompt="Extract movie details from the user text.",
)

print("=== With Rich Descriptions ===")
text1 = "I just watched The Matrix by the Wachowskis. It came out in 1999. Groundbreaking sci-fi action film."
result_rich = await agent_rich.run(text1)
print(f"Summary: {result_rich.output.short_summary}")
print(f"Length: {len(result_rich.output.short_summary.split())} words")
print(f"Year: Set in roughly {result_rich.output.year}")
print("\nNotice how the rich description lets us guide the LLM to another interpretation of the year!")

## 🧪 Practical Exercise: "The Unstructured Data Cleaner"
**Goal:**  
You are processing a queue of raw customer support emails. We'd like you to convert messy text into a structured SupportTicket objects.

**Requirements:**  
Create a Pydantic SupportTicket object meeting the following spec
* customer_name: str (Capitalized)
* sentiment_score: float (0.0 to 1.0, where 0 is angry, 1 is happy)
* severity: str (Enum: 'Low', 'Medium', 'High')
* actionable_items: List[str] (Bullet points of what to do)
* If severity is 'High' AND sentiment_score is < 0.3, the actionable_items list cannot be empty.
    * If it is empty, raise a ModelRetry (or ValueError) instructing the LLM to "Infer actionable items based on the angry tone."

**Example:**  
> SUBJECT: URGENT!!!!  
> FROM: karen.smith@email.com
>  
> I am absolutely furious. I have been on hold for 4 hours.   
> Your system charged me double for the subscription and then locked me out of my account.  
> Fix this NOW or I am suing.

In [ ]:
class Severity(str, Enum):
    LOW = "Low"
    MEDIUM = "Medium"
    HIGH = "High"


class SupportTicket(BaseModel):
    # TODO: Add fields here.

    # TODO: Add @model_validator(mode='after')
    # If severity is HIGH and sentiment is low, ensure actionable_items is not empty.
    # raise ValueError("High severity angry tickets must have actionable items!")
    pass


# TODO: Initialize Agent with result_type=SupportTicket
# cleaner_agent = ...

In [ ]:
messy_email = """
SUBJECT: URGENT!!!!
FROM: karen.smith@email.com

I am absolutely furious. I have been on hold for 4 hours. 
Your system charged me double for the subscription and then locked me out of my account.
Fix this NOW or I am suing.
"""

In [ ]:
# TODO: Run agent on messy_email
# TODO: Inspect the messages to verify the retry logic is working.

## Down the rabbit hole
In Pydantic, the choice of validator mode is essentially a choice between pre-processing incoming data and verifying the fully validated model. See [the docs](https://docs.pydantic.dev/latest/concepts/validators/) for more, but we'll go into a little exploration here. 

#### Why we have `field_validator` and `model_validator`
* `field_validator` applies to single fields in your model (noting that these can be nested Pydantic models!). It takes the value to be validated, and returns the validated value, which can be coerced (turned into another type) or mutated (modified). Note there are two ways to apply these validators to your models - **Annotations** and **decorators**.
* `model_validator` runs on your whole model, and is used when you want to compare one field against another.

#### Understanding `mode='before'` vs `mode='after'`
* `mode='before'` runs before Pydantic parses/coerces types and before the model exists. It receives the raw input (often a `dict`, but technically it can be any object passed to model validation). This is the right place to reshape, normalize, or “clean” input so that normal field parsing/validation can succeed.
* `mode='after'` (the default) runs after Pydantic has created the model instance and all fields have been individually parsed/validated. This is primarily used for logic and cross-field validation—rules that depend on relationships between multiple fields (e.g., ensuring `end_date > start_date`, or `password` matches `password_confirm`).

#### Method form (`cls` vs `self`) and deprecation
* Because `mode='before'` runs before an instance exists, it must be a `@classmethod` (using `cls`), since there is no `self` yet.
* For `mode='after'` in **model validators**, you should use a regular instance method (`self`). As of Pydantic v2.12, defining an “after” model validator as a `@classmethod` is deprecated and will emit a `PydanticDeprecatedSince212` warning. In “after” mode, operate on `self` (the validated model instance) and typically return `self`.


In [ ]:
import re
from typing import Any, Self

from pydantic import model_validator

YEAR_PATTERN = re.compile(r"The year is (\d+)")
TITLE_YEAR_PATTERN = re.compile(r"(^.*?)\((\d{4})\)$")

In [ ]:
class MovieExtractionV3(BaseModel):
    title: str
    director: str
    year: Optional[int] = None
    genres: List[Genre]
    short_summary: str = Field(description="A one-sentence summary")

    @field_validator("year", mode="after")  # After is the default option
    @classmethod
    def validate_year_not_future(cls, v: Optional[int]) -> Optional[int]:
        """Ensure the film wasn't released in the future - some logic we wish to apply."""
        if v is None:
            return v
        current_year = datetime.now().year
        if v < current_year:
            raise ValueError("Hello from after!")
        return v

    @field_validator("year", mode="before")
    @classmethod
    def maybe_extract_year(cls, v: Optional[int]) -> Optional[int]:
        """If the provided year is a string tries to extract the year from it. Note that this happens before the
        inbuilt type conversion happens."""
        if v is None:
            return v
        if isinstance(v, str):  ## We're converting the string to an int
            match = YEAR_PATTERN.match(v)
            if match:
                v = int(match.group(1))
            else:
                raise ValueError("Year not found in string")

        return v

    @model_validator(mode="before")
    @classmethod
    def mutate_combined_title_year(cls, data: Any) -> Any:  # Takes cls, returns Any
        """Some records provide metadata about the year in the title, lets handle that."""
        if "year" not in data:  # This method is pretty brittle (sorry MLEs), but it's an example
            match = TITLE_YEAR_PATTERN.match(data["title"])
            if match:
                data["title"] = match.group(1)
                data["year"] = int(match.group(2))
            else:
                raise ValueError("Year not found in title")
        return data

    @model_validator(mode="after")  # After is the default option
    def ensure_total_words_in_limit(
        self,
    ) -> Self:  # Note this works with self and not cls
        """Ensure the total words in the summary are within a reasonable limit"""
        total_words = 0
        for field in ["title", "director", "short_summary"]:
            total_words += len(getattr(self, field).split())

        if total_words > 100:
            raise ValueError("Too many words in total")
        return self


dict_data = {
    "title": "The Matrix",
    "director": "The Wachowskis",
    "year": "The year is 2999",  ## <--- Note this is now a nasty string
    "genres": [Genre.SCI_FI, Genre.ACTION],
    "short_summary": "Groundbreaking sci-fi action film.",
}

## The before validator will convert the string to an int, but the after validator will raise a ValueError. Note that
## Pydantic will usually attempt to coerce types.
test_movie = MovieExtractionV3(**dict_data)
rprint(test_movie)

unusual_dict_data = {
    "title": "The Matrix (2999)",  ## <--- Note year in title
    "director": "The Wachowskis",
    "genres": [Genre.SCI_FI, Genre.ACTION],
    "short_summary": "Groundbreaking sci-fi action film.",
}
test_unusual_movie = MovieExtractionV3(**unusual_dict_data)
rprint(test_unusual_movie)

## Bonus!
Other things I didn't know could be done in Pydantic that might prove useful patterns.
1. Raising ModelRetry errors to let the LLM know what corrections need to be made. 
2. Being able to provide two potential output classes using the Union operator, useful when there are two valid outputs such as a SuccessfulResponse or MissingData
3. Tool level validation - being able to validate the inputs to tools the Agent wants to use.

### Bonus 1 - ModelRetry
Standard Pydantic raises a ValueError which stops execution or sends a generic error. ModelRetry is a special PydanticAI exception that intercepts the error and sends a specific, instructional message back to the LLM to trigger a correction loop. In the weird world of LLM programming, it turns validation into a "conversation."

It can be used in @field_validator, @model_validator, or inside Tools themselves (we'll get to tools in a bit).

In [ ]:
from pydantic_ai import ModelRetry


# Inside a validator
@field_validator("sql_query")
def validate_sql(cls, v):
    if "DROP TABLE" in v.upper():
        # The LLM sees this message and tries again immediately
        raise ModelRetry("Dangerous command detected. Rewrite the query to be read-only.")
    return v

### Bonus 2 - Polymorphic Validation (Unions)
Often, an agent needs to decide between two different valid outcomes (e.g., "I found the answer" vs. "I need more info"). You can enforce this decision structure using Union rather than trying to cram everything into a single class.

```python
from typing import Union
from pydantic import BaseModel

class SuccessResult(BaseModel):
    answer: str
    confidence: float

class MissingInfo(BaseModel):
    missing_fields: list[str]
    question_to_user: str

# The Agent must return ONE of these two strict shapes
agent = Agent(
    ..., 
    result_type=Union[SuccessResult, MissingInfo]
)
```

### Bonus 3 - Tool-Level Validation
Validation isn't just for the output of the agent; it's also for the inputs to the tools. If an agent calls a tool with valid types (e.g., date="2024-01-01") but invalid logic (e.g., that date is a holiday), the tool itself can reject the call. Useful with "expensive" tools where you really want to make sure you're only putting in high quality outputs.

```python
@agent.tool
def book_meeting(ctx, date: str):
    if is_holiday(date):
        # This acts as a validator that feeds back to the LLM
        raise ModelRetry(f"{date} is a holiday. Please pick a weekday.")
    return "Meeting booked"
```